In [ ]:
from pathlib import Path

GCD_PATH = Path("/project/6008051/pone_simulation/GCD_Library/PONE_800mGrid.i3.gz")

# Geometry-selection training output root. The notebook will search below this
# for learned_string_selection.csv files produced by train_geometry_selection.py.
RESULT_ROOT = Path(
    "/project/def-nahee/kbas/Graphnet-Applications/Results/340StringMC/"
    "full_geometry_emax1e6/geometry_selection"
)

# Optional: replace with explicit CSV paths if you only want specific runs.
SELECTION_CSVS = sorted(RESULT_ROOT.rglob("learned_string_selection.csv"))

# If the final learned CSVs are not written yet, this stays empty for now.
SELECTION_CSVS

In [ ]:
from collections import defaultdict

import pandas as pd
import matplotlib.pyplot as plt
from icecube import dataio


def read_geometry(gcd_path):
    """Return one x/y point per string from the GCD geometry."""
    gcd_path = Path(gcd_path)
    if not gcd_path.exists():
        raise FileNotFoundError(f"GCD file not found: {gcd_path}")

    f = dataio.I3File(str(gcd_path), "r")
    geo_frame = None
    while f.more():
        frame = f.pop_frame()
        if frame.Stop == frame.Geometry:
            geo_frame = frame
            break
    f.close()

    if geo_frame is None:
        raise RuntimeError(f"No Geometry frame found in {gcd_path}")

    xy_by_string = defaultdict(list)
    for omkey, omgeo in geo_frame["I3Geometry"].omgeo.items():
        xy_by_string[int(omkey.string)].append((float(omgeo.position.x), float(omgeo.position.y)))

    rows = [
        {"string_id": string_id, "x": positions[0][0], "y": positions[0][1]}
        for string_id, positions in sorted(xy_by_string.items())
    ]
    return pd.DataFrame(rows)


def read_selection(csv_path):
    """Read model selection CSV or a flat comma-separated string-id CSV."""
    csv_path = Path(csv_path)
    df = pd.read_csv(csv_path)

    if {"string_id", "selected_topk"}.issubset(df.columns):
        selected = df[df["selected_topk"].astype(bool)].copy()
        selected["string_id"] = selected["string_id"].astype(int)
        if "gate_score" not in selected.columns:
            selected["gate_score"] = 1.0
        if "rank" not in selected.columns:
            selected["rank"] = range(1, len(selected) + 1)
        return selected.sort_values("rank")

    # Legacy geometry CSVs in this folder are often flat comma-separated IDs.
    text = csv_path.read_text().strip()
    ids = []
    for line in text.splitlines():
        for item in line.split(","):
            item = item.strip()
            if item:
                ids.append(int(item))
    ids = sorted(set(ids))
    return pd.DataFrame({"string_id": ids, "gate_score": 1.0, "rank": range(1, len(ids) + 1)})


def label_from_path(csv_path):
    parts = Path(csv_path).parts
    pieces = []
    for key in ("first_category", "second_category"):
        if key in parts:
            pieces.append(key)
    pieces.extend([p for p in parts if p.startswith("class")])
    if "train_and_val" in parts:
        i = parts.index("train_and_val")
        if i + 1 < len(parts):
            pieces.append(parts[i + 1])
    exp = next((p for p in parts if p.startswith("exp")), None)
    if exp:
        pieces.append(exp)
    return " / ".join(pieces) if pieces else Path(csv_path).parent.name


def set_equal_limits(ax, full_df, pad=0.05):
    xmin, xmax = full_df["x"].min(), full_df["x"].max()
    ymin, ymax = full_df["y"].min(), full_df["y"].max()
    span = max(xmax - xmin, ymax - ymin)
    xmid = 0.5 * (xmin + xmax)
    ymid = 0.5 * (ymin + ymax)
    half = 0.5 * span * (1 + pad)
    ax.set_xlim(xmid - half, xmid + half)
    ax.set_ylim(ymid - half, ymid + half)
    ax.set_aspect("equal", adjustable="box")

In [ ]:
full_geometry = read_geometry(GCD_PATH)
print(f"Full geometry strings: {len(full_geometry)}")

selections = []
for csv_path in SELECTION_CSVS:
    selected = read_selection(csv_path)
    plot_df = full_geometry.merge(selected, on="string_id", how="inner")
    missing = sorted(set(selected["string_id"]) - set(plot_df["string_id"]))
    if missing:
        print(f"Warning: {csv_path} has {len(missing)} string IDs not found in GCD: {missing[:10]}")
    selections.append({"path": csv_path, "label": label_from_path(csv_path), "df": plot_df})

print(f"Selection CSVs found: {len(selections)}")
[(item["label"], len(item["df"])) for item in selections]

In [ ]:
selection_sets = {
    item["label"]: set(item["df"]["string_id"].astype(int))
    for item in selections
}

summary_rows = []
for item in selections:
    df = item["df"]
    summary_rows.append({
        "selection": item["label"],
        "n_selected": len(df),
        "min_string": int(df["string_id"].min()) if len(df) else None,
        "max_string": int(df["string_id"].max()) if len(df) else None,
        "mean_gate_score": float(df["gate_score"].mean()) if len(df) else None,
        "min_gate_score": float(df["gate_score"].min()) if len(df) else None,
        "max_gate_score": float(df["gate_score"].max()) if len(df) else None,
    })

selection_summary = pd.DataFrame(summary_rows)
selection_summary

In [ ]:
labels = list(selection_sets)

overlap_counts = pd.DataFrame(index=labels, columns=labels, dtype=int)
for a in labels:
    for b in labels:
        overlap_counts.loc[a, b] = len(selection_sets[a] & selection_sets[b])

overlap_counts

In [ ]:
jaccard = pd.DataFrame(index=labels, columns=labels, dtype=float)
for a in labels:
    for b in labels:
        union = selection_sets[a] | selection_sets[b]
        jaccard.loc[a, b] = len(selection_sets[a] & selection_sets[b]) / len(union) if union else 0.0

jaccard

In [ ]:
if selection_sets:
    common_all = set.intersection(*selection_sets.values())
else:
    common_all = set()

common_all_df = full_geometry[full_geometry["string_id"].isin(common_all)].sort_values("string_id")
print(f"Strings common to all selections: {len(common_all_df)}")
common_all_df

In [ ]:
from collections import Counter

frequency = Counter()
for selected_set in selection_sets.values():
    frequency.update(selected_set)

frequency_df = pd.DataFrame(
    [{"string_id": string_id, "n_selections": count} for string_id, count in frequency.items()]
).sort_values(["n_selections", "string_id"], ascending=[False, True])

frequency_df = frequency_df.merge(full_geometry, on="string_id", how="left")
frequency_df.head(40)

In [ ]:
if frequency_df.empty:
    print("No selections available yet.")
else:
    fig, ax = plt.subplots(figsize=(10, 9))
    freq_plot = full_geometry.merge(
        frequency_df[["string_id", "n_selections"]],
        on="string_id", how="left"
    )
    freq_plot["n_selections"] = freq_plot["n_selections"].fillna(0).astype(int)

    bg = freq_plot[freq_plot["n_selections"] == 0]
    fg = freq_plot[freq_plot["n_selections"] > 0]

    ax.scatter(bg["x"], bg["y"], s=18, c="#d6d6d6", alpha=0.45, linewidths=0)
    points = ax.scatter(
        fg["x"], fg["y"],
        s=35 + 22 * fg["n_selections"],
        c=fg["n_selections"], cmap="magma_r",
        edgecolors="#111111", linewidths=0.45,
    )

    for _, row in fg.iterrows():
        if row["n_selections"] == fg["n_selections"].max():
            ax.text(row["x"], row["y"], str(int(row["string_id"])), fontsize=6,
                    ha="left", va="bottom", color="#111111")

    ax.set_title("String selection frequency across all runs")
    ax.set_xlabel("X (m)")
    ax.set_ylabel("Y (m)")
    ax.grid(True, alpha=0.18)
    set_equal_limits(ax, full_geometry)
    cbar = fig.colorbar(points, ax=ax, fraction=0.035, pad=0.02)
    cbar.set_label("number of selections")
    plt.tight_layout()
    plt.show()

In [ ]:
if not selections:
    print("No learned_string_selection.csv files found yet. Run this cell again after training finishes.")
else:
    n = len(selections)
    fig, axes = plt.subplots(n, 1, figsize=(9, 8 * n), squeeze=False)
    axes = axes[:, 0]

    for ax, item in zip(axes, selections):
        selected_df = item["df"]
        unselected_df = full_geometry[~full_geometry["string_id"].isin(selected_df["string_id"])]

        ax.scatter(
            unselected_df["x"], unselected_df["y"],
            s=18, c="#c9c9c9", alpha=0.55, linewidths=0,
            label="Full geometry"
        )
        points = ax.scatter(
            selected_df["x"], selected_df["y"],
            s=70, c=selected_df["gate_score"], cmap="viridis",
            edgecolors="#111111", linewidths=0.45,
            label=f"Selected ({len(selected_df)})"
        )

        for _, row in selected_df.iterrows():
            ax.text(row["x"], row["y"], str(int(row["string_id"])), fontsize=5,
                    ha="left", va="bottom", color="#111111")

        ax.set_title(item["label"], fontsize=14)
        ax.set_xlabel("X (m)")
        ax.set_ylabel("Y (m)")
        ax.grid(True, alpha=0.18)
        ax.legend(loc="upper right")
        set_equal_limits(ax, full_geometry)

        cbar = fig.colorbar(points, ax=ax, fraction=0.035, pad=0.02)
        cbar.set_label("gate score")

    fig.suptitle("Learned geometry-selection subselections", fontsize=18, y=0.995)
    plt.tight_layout()
    plt.show()